In [1]:
# ============================================================
# ConversationChain → RunnableWithMessageHistory
# ============================================================
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv

load_dotenv()

chat = ChatOpenAI(model="gpt-3.5-turbo-0125")
history = InMemoryChatMessageHistory()
chain = RunnableWithMessageHistory(chat, lambda session_id: history)
config = {"configurable": {"session_id": "sessao-1"}}

def predict(input: str):
    resposta = chain.invoke(
        [HumanMessage(content=input)],
        config=config
    )
    return resposta.content

print(predict("Oi"))
print(predict("Olá novamente"))

Oi! Como posso ajudar você hoje?
Olá! Estou aqui para ajudar. Como posso ser útil desta vez?


In [2]:
# ============================================================
# ConversationChain com PromptTemplate → ChatPromptTemplate
# ============================================================
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "Essa é uma conversa amigável entre um humano e uma IA"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain_com_prompt = RunnableWithMessageHistory(
    prompt | chat,
    lambda session_id: history,
    input_messages_key="input",
    history_messages_key="history"
)

def predict_com_prompt(input: str):
    resposta = chain_com_prompt.invoke(
        {"input": input},
        config=config
    )
    return resposta.content

print(predict_com_prompt("Oi"))

Olá de novo! Tem alguma dúvida ou algo em que eu possa te ajudar?


In [3]:
# ============================================================
# LLMChain → prompt | chat (LCEL)
# ============================================================
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

chat = ChatOpenAI(model="gpt-3.5-turbo-0125")
produto = "LLMs com LangChain"

prompt_nome = PromptTemplate.from_template("""
    Escolha o melhor nome para mim sobre uma empresa que desenvolve soluções em {produto}
""")

chain_nome = prompt_nome | chat | StrOutputParser()

print(chain_nome.invoke({"produto": produto}))

Optilink Solutions at LangChain


In [4]:
# ============================================================
# SimpleSequentialChain → chain1 | chain2 (LCEL)
# ============================================================
prompt_descricao = PromptTemplate.from_template("""
    Sobre a empresa com nome {nome_empresa}.
    Informe uma descrição de até 20 palavras.
""")

chain_descricao = prompt_descricao | chat | StrOutputParser()

# Encadeando diretamente com pipe
chain_sequencial = (
    prompt_nome
    | chat
    | StrOutputParser()
    | (lambda nome: {"nome_empresa": nome})
    | chain_descricao
)

print(chain_sequencial.invoke({"produto": produto}))

A LangSolutions Tech é uma empresa especializada em soluções de tecnologia e linguagem, oferecendo serviços inovadores e eficientes.


In [5]:
# ============================================================
# SequentialChain → RunnablePassthrough + LCEL
# ============================================================
from langchain_core.runnables import RunnablePassthrough

prompt_traducao = PromptTemplate.from_template("""
    Crie um nome em espanhol para a empresa de nome {nome_empresa}
    que possui a seguinte descrição {descricao_empresa}
""")

chain_traducao = prompt_traducao | chat | StrOutputParser()

# Chain completo com múltiplas saídas
chain_completo = (
    RunnablePassthrough.assign(
        nome_empresa = prompt_nome | chat | StrOutputParser()
    )
    | RunnablePassthrough.assign(
        descricao_empresa = prompt_descricao | chat | StrOutputParser()
    )
    | RunnablePassthrough.assign(
        nome_espanhol = chain_traducao
    )
)

resposta = chain_completo.invoke({"produto": produto})

print(resposta["produto"])
print(resposta["nome_empresa"])
print(resposta["descricao_empresa"])
print(resposta["nome_espanhol"])

LLMs com LangChain
"LangNet Solutions"
A LangNet Solutions é uma empresa de soluções em tecnologia da informação, oferecendo serviços de desenvolvimento e consultoria.
Soluciones TecnoLangNet
